# Vuln2Action - T5 stage 1 evaluation (BLEU / ROUGE)

M?c ti?u: ??nh gi? checkpoint `checkpoint-693` c?a l?n benchmark 1 epoch tr?n
**validation split** b?ng BLEU v? ROUGE.

- Dataset: `minhiw/vuln2action-flow-114560`
- Checkpoint source: output c?a notebook `minhiw/vuln2action-t5-stage1-benchmark`
- Kh?ng c? exploit source code trong dataset.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os

flow_hits = sorted(glob.glob("/kaggle/input/**/flow_validation.jsonl", recursive=True))
print("flow_validation hits:", flow_hits)
if not flow_hits:
    raise FileNotFoundError("flow_validation.jsonl not found under /kaggle/input")

FLOW_DIR = Path(flow_hits[0]).parent
os.environ["FLOW_DIR"] = str(FLOW_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("FLOW_DIR:", FLOW_DIR)
for name in ["flow_train.jsonl", "flow_validation.jsonl", "flow_stats.json"]:
    p = FLOW_DIR / name
    print(name, p.exists(), f"{p.stat().st_size / 1024 / 1024:.1f} MiB" if p.exists() else "")

ckpt_hits = sorted(Path("/kaggle/input").rglob("checkpoint-693"))
print("checkpoint-693 hits:", [str(p) for p in ckpt_hits])
if not ckpt_hits:
    ckpt_hits = sorted(Path("/kaggle/input").rglob("checkpoint-*"))
    print("fallback checkpoints:", [str(p) for p in ckpt_hits])
if not ckpt_hits:
    raise FileNotFoundError("no T5 checkpoint found under /kaggle/input")

CKPT = ckpt_hits[-1]
os.environ["CKPT"] = str(CKPT)
print("CKPT:", CKPT)
for name in ["config.json", "model.safetensors", "pytorch_model.bin", "tokenizer.json", "spiece.model", "training_args.bin"]:
    p = CKPT / name
    print(name, p.exists(), f"{p.stat().st_size / 1024 / 1024:.1f} MiB" if p.exists() else "")


In [ ]:
!python scripts/evaluate_t5.py \
  --checkpoint "$CKPT" \
  --dataset "$FLOW_DIR" \
  --split validation \
  --target-field target_techniques \
  --output-dir /kaggle/working/t5_stage1_eval \
  --max-input-length 256 \
  --max-target-length 256 \
  --batch-size 8 \
  --num-beams 4 \
  --fp16


In [ ]:
import json
from pathlib import Path

out = Path("/kaggle/working/t5_stage1_eval")
metrics = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, indent=2, ensure_ascii=False))

print("\n--- sample predictions ---")
for i, line in enumerate((out / "predictions.jsonl").open(encoding="utf-8")):
    if i >= 5:
        break
    row = json.loads(line)
    print(f"\n[{i}] {row['cve_id']}")
    print("REF:", row["reference"][:300])
    print("PRED:", row["prediction"][:300])


## C?ch ??c k?t qu?

- BLEU: so kh?p n-gram v?i reference flow.
- ROUGE-1/2/L: ?? ph? token/c?m t? so v?i reference.
- ??y l? **validation split**, kh?ng ph?i test 100 CAPEC c?a paper.
- N?u BLEU th?p nh?ng ROUGE-L ?n, model c? th? ??ng ? nh?ng kh?c c?ch di?n ??t.
- Lu?n ??c th?m sample predictions tr??c khi k?t lu?n.
